# Amazon Nova — picking a tier, and proving the small one is enough

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Nova is Amazon's own model family and the clearest size ladder on Bedrock. It is
`bedrock-runtime` only: there is no `bedrock-mantle` path, so Converse is the
API.

| Model | Input | Tier |
|---|---|---|
| `nova-micro` | text | cheapest, text only |
| `nova-lite` | text, image, video | low cost, multimodal |
| `nova-pro` | text, image, video | higher quality, multimodal |
| `nova-2-lite` | text, image, video | generation 2, multimodal; **no bare-ID access** (see section 5) |
| `nova-premier` | — | **legacy, blocked** (see section 4) |

The interesting question with a ladder is never "which is best" — it is "what is
the cheapest tier that still passes". This notebook answers that empirically
rather than by reputation. `nova-2-lite` is a second generation rather than a
fourth rung, so it is measured alongside the ladder on every task instead of
being slotted into it.

It also addresses differently. The three generation-1 models accept a bare model
ID; `nova-2-lite` is `INFERENCE_PROFILE`-only and refuses one. Section 5 measures
that against the catalogue rather than asking you to remember it.

Nova also accepts **video** input on lite and pro. Video is out of scope for this
collection, so the vision cells use images; the same content-block shape applies.


### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `slide_jpeg` | JPEG bytes of a slide from a public AWS talk, so vision cells have a known answer |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |
| `control_client` | a boto3 `bedrock` client (model and profile catalogues) |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [ ]:
import sys

sys.path.insert(0, "../_shared")

from bedrock import (
    SLIDE_CALLOUTS,
    SLIDE_TITLE,
    converse,
    converse_tool_uses,
    endpoints_for,
    keyword_recall,
    resolve_runtime_id,
    runtime_models,
    slide_jpeg,
)

REGION = "us-east-1"
MICRO = "amazon.nova-micro-v1"
LITE = "amazon.nova-lite-v1"
PRO = "amazon.nova-pro-v1"
NOVA2_LITE = "amazon.nova-2-lite-v1"

# LADDER is the generation-1 size ladder, which is what sections 1-3 compare.
# NOVA2_LITE is a different generation, so it rides along in MODELS rather than
# being treated as a fourth rung.
LADDER = [MICRO, LITE, PRO]
MODELS = LADDER + [NOVA2_LITE]

catalogue = runtime_models(REGION)
print(f"{'model':<26} {'input':<22} {'inference types'}")
print("-" * 76)
for model in MODELS:
    entry = catalogue[model]
    print(f"{model:<26} {','.join(sorted(entry['in'])):<22} "
          f"{','.join(sorted(entry['infer']))}")

print()
print("endpoints:", {m.split(".")[-1]: endpoints_for(m, REGION) for m in [MICRO]})
print("=> bedrock-mantle is False: Nova is a runtime-only family.")


## 1. The same task at three tiers

A single easy prompt tells you nothing — every tier passes. Use a task with a
checkable answer and enough structure that a weaker model can visibly fail.

Below: extract three fields as JSON. The check is mechanical, so "did it pass"
is not a judgement call.


In [ ]:
import json as jsonlib

PROMPT = (
    "Extract to JSON with keys name, city, years. "
    "Reply with JSON only, no prose.\n\n"
    "Priya has been an engineer in Singapore for eleven years."
)
EXPECTED = {"name": "Priya", "city": "Singapore", "years": 11}


def grade(raw: str) -> str:
    """Did the model return the three fields with the right values?"""
    text = (raw or "").strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        text = text[4:] if text.startswith("json") else text
    try:
        got = jsonlib.loads(text.strip())
    except Exception:
        return "unparseable"
    hits = sum(
        1
        for key, want in EXPECTED.items()
        if str(got.get(key, "")).lower() == str(want).lower()
    )
    return f"{hits}/3 fields correct"


print(f"{'model':<26} {'tokens':>7}  {'verdict':<22} answer")
print("-" * 92)
for model in MODELS:
    text, response = converse(
        model,
        [{"role": "user", "content": [{"text": PROMPT}]}],
        max_tokens=200,
        temperature=0.0,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} {'-':>7}  ERROR {error[:40]}")
        continue
    total = response.get("usage", {}).get("totalTokens", 0)
    print(f"{model:<26} {total:>7}  {grade(text):<22} "
          f"{text.strip()[:34].replace(chr(10), ' ')}")


## 2. Vision on lite and pro

`nova-micro` is text-only, so sending it an image is a design error rather than a
quality question. The three multimodal models get the same generated image with a
known answer, so "did it look" is verifiable.


In [ ]:
jpeg = slide_jpeg()
QUESTION = "Read this slide. Give its title, then quote the three green callout lines."
print(f"slide: {len(jpeg)} bytes of JPEG; title is {SLIDE_TITLE!r}\n")

for model in (LITE, PRO, NOVA2_LITE):
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [
                    {"image": {"format": "jpeg", "source": {"bytes": jpeg}}},
                    {"text": QUESTION},
                ],
            }
        ],
        max_tokens=220,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} ERROR {error[:60]}")
        continue
    hits, total = keyword_recall(text, SLIDE_CALLOUTS)
    title_seen = SLIDE_TITLE.lower() in (text or "").lower()
    print(f"{model:<26} callouts {hits}/{total}  title={title_seen}")
    print(f"{'':<26} {' '.join(text.split())[:90]!r}")

# And the design error, so you recognise it.
text, response = converse(
    MICRO,
    [
        {
            "role": "user",
            "content": [
                {"image": {"format": "jpeg", "source": {"bytes": jpeg}}},
                {"text": QUESTION},
            ],
        }
    ],
    max_tokens=40,
    region=REGION,
)
error = (response.get("error") or {}).get("message")
print(f"\n{MICRO} (text-only) with an image:")
print("   ", error[:130] if error else f"unexpectedly accepted: {text.strip()[:60]}")

## 3. Tool use across the ladder

Tool support is not a given at the cheapest tier, so check it rather than assume.
The assertion here is on the *arguments*, not on whether a call happened — a tool
call with wrong operands still reports `stopReason: tool_use`.


In [ ]:
TOOLS = [
    {
        "toolSpec": {
            "name": "convert_currency",
            "description": "Convert an amount between two currencies",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "amount": {"type": "number"},
                        "from": {"type": "string"},
                        "to": {"type": "string"},
                    },
                    "required": ["amount", "from", "to"],
                }
            },
        }
    }
]

print(f"{'model':<26} {'stop':<12} tool call")
print("-" * 78)
for model in MODELS:
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [{"text": "Convert 250 SGD to JPY. Use the tool."}],
            }
        ],
        max_tokens=400,
        tools=TOOLS,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} ERROR {error[:44]}")
        continue
    uses = converse_tool_uses(response)
    if not uses:
        print(f"{model:<26} {response.get('stopReason'):<12} (no tool call)")
        continue
    args = uses[0]["input"]
    ok = (
        str(args.get("amount")) in {"250", "250.0"}
        and str(args.get("from", "")).upper() == "SGD"
        and str(args.get("to", "")).upper() == "JPY"
    )
    print(f"{model:<26} {response.get('stopReason'):<12} "
          f"{args}  {'correct' if ok else 'ARGS WRONG'}")


## 4. `nova-premier` is legacy, and Bedrock tells you so

`nova-premier` is still in the catalogue, which is not the same as being callable.
Bedrock marks superseded models as legacy and refuses new usage. The error is
specific and worth recognising, because the same wording gates several older
models across providers.

This is why "read the catalogue" is not sufficient on its own: the catalogue lists
what exists, not what you are allowed to invoke today.


In [5]:
from bedrock import control_client, runtime_client

PREMIER = "amazon.nova-premier-v1"

# The catalogue is perfectly happy about it.
entry = catalogue.get(PREMIER)
print(f"in catalogue      : {PREMIER in catalogue}")
print(f"inference types   : {sorted(entry['infer']) if entry else '-'}")

# So is the entitlement check.
availability = control_client(REGION).get_foundation_model_availability(
    modelId=f"{PREMIER}:0"
)
print("authorizationStatus:", availability.get("authorizationStatus"))
print("entitlement        :", availability.get("entitlementAvailability"))

# And yet:
print("\nactually calling it:")
try:
    runtime_client(REGION).converse(
        modelId=resolve_runtime_id(PREMIER, REGION),
        messages=[{"role": "user", "content": [{"text": "hi"}]}],
        inferenceConfig={"maxTokens": 12},
    )
    print("    accepted")
except Exception as exc:
    print(f"    {type(exc).__name__}")
    print(f"    {str(exc)[-160:]}")


in catalogue      : True
inference types   : ['INFERENCE_PROFILE']


authorizationStatus: AUTHORIZED
entitlement        : AVAILABLE

actually calling it:


    ResourceNotFoundException
    is Model is marked by provider as Legacy and you have not been actively using the model in the last 30 days. Please upgrade to an active model on Amazon Bedrock


## 5. `nova-2-lite` has no bare-ID access, and the catalogue predicts it

Section 4 showed that catalogue presence is not permission. This section asks a
narrower question: when the catalogue says a model supports `ON_DEMAND`, does a
**bare model ID** actually work, and when it says `INFERENCE_PROFILE` only, is the
bare ID actually refused?

That matters because it is the difference between working and 400 for the same
code. The three generation-1 models here carry `ON_DEMAND`; `nova-2-lite` does
not, so the call pattern that works for `nova-lite` fails for `nova-2-lite`:

    Invocation of model ID amazon.nova-2-lite-v1:0 with on-demand throughput
    isn't supported. Retry your request with the ID or ARN of an inference
    profile that contains this model.

`resolve_runtime_id()` is what the rest of this collection uses to avoid that; it
prefers the geo-prefixed profile whenever one exists. The cell below calls each
model **twice** — once with the bare catalogue ID and once with the resolved ID —
and counts how often the catalogue flag agreed with what the service did. Read the
count: if it is not 4/4, the flag is not a reliable predictor and the table above
it says which model broke it.

In [ ]:
HELLO = [{"role": "user", "content": [{"text": "Reply with the single word: ok"}]}]


def call_verdict(model_id: str) -> str:
    """Call one exact model ID with no resolution, and report what came back.

    resolve=False is the point of this cell: resolve_runtime_id() would rewrite a
    bare ID into a profile ID and hide the very difference being measured.
    """
    _, response = converse(
        model_id, HELLO, max_tokens=16, region=REGION, resolve=False
    )
    error = (response.get("error") or {}).get("message")
    return "accepted" if not error else f"refused ({error.split('.')[0][:52]})"


print(f"{'model':<26} {'ON_DEMAND':<10} {'bare ID':<24} resolved ID")
print("-" * 100)
agreed = 0
for model in MODELS:
    entry = catalogue[model]
    on_demand = "ON_DEMAND" in entry["infer"]
    bare = call_verdict(entry["id"])
    resolved_id = resolve_runtime_id(model, REGION)
    resolved = call_verdict(resolved_id)
    # The claim under test: does the ON_DEMAND flag predict the bare-ID result?
    if on_demand == (bare == "accepted"):
        agreed += 1
    print(f"{model:<26} {str(on_demand):<10} {bare[:24]:<24} {resolved_id}")
    print(f"{'':<26} {'':<10} {'':<24} {resolved}")

print()
print(f"ON_DEMAND flag agreed with the bare-ID result: {agreed}/{len(MODELS)}")
print(f"bare IDs that were refused: "
      f"{[m for m in MODELS if 'ON_DEMAND' not in catalogue[m]['infer']]}")
print("=> resolve_runtime_id() is not a convenience for these; it is required.")


## Takeaways

- **Nova is `bedrock-runtime` only.** No `bedrock-mantle` path, so no bearer token
  and no OpenAI-shaped option.
- **Pick the tier with a graded task, not a vibe.** Section 1 gives a mechanical
  pass mark; if `nova-micro` scores 3/3 on your real task, the higher tiers are
  spend without return.
- **`nova-micro` is text-only.** Sending it an image is a design error, not a
  quality trade-off.
- **Assert tool arguments.** `stopReason: tool_use` only says a call was made, not
  that it was right.
- **Catalogue presence is not permission.** `nova-premier` is listed and shows as
  authorised, and still refuses every call because the provider marked it legacy.
  Sweep your intended model list with one cheap call before you design around it.
- **`nova-2-lite` needs an inference profile.** It is `INFERENCE_PROFILE`-only, so
  the bare ID that works for `nova-lite` returns a 400. Section 5 measures the
  bare and resolved forms side by side; pass model IDs through
  `resolve_runtime_id()` rather than hardcoding either form.
